# Dataset EDA, fairness checks, and classifier training

Companion to `ref material/BUILD.md` S5.3–S5.5. This notebook does three things, in
this order, and the order matters:

1. **Audit the dataset before training on it.** Balance, length, vocabulary
   overlap, and the flag ground truth.
2. **Test whether the dataset is fair** — that is, whether a classifier could
   score well for the wrong reason. The leakage check is the one that would
   invalidate every downstream number if it failed, so it runs before training.
3. **Fine-tune MiniLM-L6** and write the artefacts the evaluation reads.

**Nothing here is a result.** Results live in `eval/results/`, produced by
`eval/run_eval.py` on the held-out test split. This notebook is where the dataset
is interrogated and the model is built; reporting happens elsewhere, deliberately,
so a number can never come from the same pass that fitted it.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The notebook lives in models/, so the repository root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

SEED = 42
np.random.seed(SEED)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(f"root: {ROOT}")

---
## 1. Load the corpus and the generated emails

The emails are frozen and committed, so this notebook never regenerates them. If
`emails.jsonl` is absent, run `python scripts/generate_emails.py` first — that step
uses a paid API and is deliberately not on the reproduction path.

In [ ]:
from triage.sop.index import build_index

index = build_index()
print(f"SOPs: {len(index.sops)} ({len(index.indexed)} indexed, {len(index.held_out)} held out)")

EMAILS = ROOT / "data" / "generated" / "emails.jsonl"
if not EMAILS.exists():
    raise SystemExit(
        f"{EMAILS.relative_to(ROOT)} not found.\n"
        "Run: python scripts/generate_emails.py"
    )

df = pd.DataFrame([json.loads(line) for line in EMAILS.read_text(encoding="utf-8").splitlines()])
df["n_chars"] = df["body"].str.len()
df["n_words"] = df["body"].str.split().str.len()
print(f"{len(df):,} emails, {df['label'].nunique()} classes")
df.head(3)


---
## 4. The fairness checks: is this corpus honest?

**This is the section that would invalidate everything downstream.**

The scenarios describe *situations* and never name a class, so the generator cannot
encode the answer in the wording. But intent is not evidence, so seven checks look
for the evidence. All are implemented in `scripts/check_leakage.py` and run before
training, not after.

### What the committed corpus scored

| Check | Result | Reading |
|---|---|---|
| Balance | 150/class exactly | equal statistical power per topic |
| **Vocabulary (TF-IDF + logreg)** | **0.754** | in band: chance 0.083, ceiling 0.95 |
| System vocabulary | none found | no email reads as though it knows the taxonomy |
| Single-class confinement | 19 words, all topic nouns | no generator tell |
| Length-only classifier | 0.118 | ≈ chance (0.083) |
| Near-duplicates | 8 pairs > 0.9 | 0.4% of the corpus |
| Flag confounding | 6 classes each | not a proxy for the classifier |

### How to read 0.754

A bag of words with no semantics separates the classes three quarters of the time.
That is what an honest corpus looks like:

| Score | Meaning |
|---|---|
| ~0.08 | chance — the text carries no class signal, implausible for real language |
| **0.5–0.8** | **expected.** Topical words genuinely differ; that IS the task |
| > 0.95 | **a problem.** A giveaway token exists; rebuild the corpus |

"instalment" really does indicate a payment enquiry. A classifier is *supposed* to
use that. What would be fatal is near-perfect separation, which would mean the
generator left a tell — and the encoder would learn the tell rather than the
citizen's problem.

### A check that was wrong, and what replaced it

The first version failed 121 emails for containing "rental income", "tax relief" or
"foreign income", on the reasoning that these are class names.

That was a mistake. A landlord writing about a spare room says "rental income",
because that is the ordinary English name for the thing. The check was conflating
*the class is named after its topic* with *the generator leaked the label*.

Two pieces of evidence settled it:

1. None of those phrases appears anywhere in the scenario text the generator saw.
2. Words like "relief" and "overseas" bleed across class boundaries the way real
   vocabulary does — "relief" appears in relief, hardship and foreign-income emails
   alike — rather than sitting confined to one class.

It was replaced by two better checks: a list of **system vocabulary** a member of
the public would never write ("triage", "escalation bucket", "confidence score"),
and a **data-driven confinement scan** that finds candidate tells without needing a
hand-written list of them.


In [ ]:
counts = df["label"].value_counts().sort_index()
print(counts.to_string())
print(f"\nmin {counts.min()}  max {counts.max()}  spread {counts.max() - counts.min()}")
assert counts.max() - counts.min() <= 1, "classes are not balanced"

fig, ax = plt.subplots(figsize=(9, 4))
counts.plot.barh(ax=ax, color="#4C78A8")
ax.set_xlabel("emails"); ax.set_title("Class balance (full corpus)")
plt.tight_layout(); plt.show()

In [ ]:
# Balance WITHIN each split. A stratified split should hold, but a silent failure
# here would put a class below the count its published metric assumes.
if "split" in df.columns:
    pivot = df.pivot_table(index="label", columns="split", values="id", aggfunc="count")
    pivot = pivot[["train", "calibration", "test"]]
    pivot["total"] = pivot.sum(axis=1)
    display(pivot)
    print(f"\nminimum test examples in any class: {pivot['test'].min()}")
    assert pivot["test"].min() >= 40, "a class is below the ~45 the reporting assumes"
else:
    print("no split column yet - run scripts/make_splits.py")

---
## 3. Length and style

Two questions:

- Is any class systematically longer or shorter? A length signal is a legitimate
  feature, but if it *alone* separated the classes the model could ignore the text.
- Did the style axis actually apply? Standard, imperfect and Singlish registers were
  requested; if the generator flattened them, the corpus is less varied than the
  writeup claims.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["n_words"].plot.hist(bins=50, ax=axes[0], color="#4C78A8")
axes[0].set_xlabel("words"); axes[0].set_title("Email length")

order = df.groupby("label")["n_words"].median().sort_values().index
df.boxplot(column="n_words", by="label", ax=axes[1], vert=False, grid=False)
axes[1].set_title("Length by class"); axes[1].set_xlabel("words")
plt.suptitle(""); plt.tight_layout(); plt.show()

print(df["n_words"].describe().to_string())

In [ ]:
# Length must not be a giveaway on its own: if a one-feature model separates the
# classes well, the generator encoded the label in how much it wrote.
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

X_len = df[["n_words", "n_chars"]].to_numpy()
y = df["label"].to_numpy()

baseline = cross_val_score(DummyClassifier(strategy="most_frequent"), X_len, y, cv=5).mean()
length_only = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=SEED),
                              X_len, y, cv=5).mean()
print(f"majority-class baseline : {baseline:.3f}")
print(f"length-only classifier  : {length_only:.3f}  (chance = {1/df['label'].nunique():.3f})")
if length_only > 0.25:
    print("\nWARNING: length alone is unexpectedly predictive - inspect generation.")

In [ ]:
for axis in ("literacy", "register", "season", "persona_role"):
    if axis in df.columns:
        print(f"\n{axis}:")
        print((df[axis].value_counts(normalize=True) * 100).round(1).to_string())

---
## 4. The fairness check that matters: vocabulary leakage

**This is the check that would invalidate everything downstream.**

The scenarios describe *situations*, never labels, precisely so the generator cannot
encode the answer in the wording. But intent is not proof. The test is empirical:
fit TF-IDF + logistic regression — a bag of words with no semantics at all — and see
how well it separates the classes.

Reading the result:

| Accuracy | Interpretation |
|---|---|
| ~0.08 | chance. The text carries no class signal — implausible for real language |
| 0.5–0.8 | **expected.** Words genuinely differ by topic; that is the task |
| >0.95 | **a problem.** A bag of words should not be near-perfect on natural email |

A very high score means a giveaway token exists, and the encoder would be learning
the generator's tell rather than the citizen's problem. The corpus would be rebuilt.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline

CHANCE = 1 / df["label"].nunique()
CEILING, FLOOR = 0.95, 0.30

leak = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=2000, random_state=SEED),
)
pred = cross_val_predict(leak, df["body"], y, cv=5)
acc = (pred == y).mean()

print(f"TF-IDF + logistic regression : {acc:.3f}")
print(f"chance                       : {CHANCE:.3f}")
print(f"failure ceiling              : {CEILING}")

if acc > CEILING:
    print("\n*** LEAKAGE — a giveaway token exists. Inspect features and rebuild. ***")
elif acc < FLOOR:
    print("\nUnusually low: the classes may be under-separated.")
else:
    print("\nIn the expected band: topical vocabulary differs, no giveaway.")


In [ ]:
# What is the bag of words actually keying on? A token that IS the label - or a
# generator artefact - shows up here immediately.
leak.fit(df["body"], y)
vec, clf = leak.steps[0][1], leak.steps[1][1]
names = np.array(vec.get_feature_names_out())

for i, label in enumerate(clf.classes_):
    top = names[np.argsort(clf.coef_[i])[-8:]][::-1]
    print(f"{label:26} {', '.join(top)}")

In [ ]:
# The confusion matrix of the WEAK model is diagnostic: pairs it confuses are the
# genuinely hard ones, and should match the pairs the design predicted
# (account_specific vs its topic; the two out-of-scope classes; FIL vs ASM).
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y, pred, ax=ax, xticks_rotation=90, colorbar=False, normalize="true", values_format=".2f",
)
ax.set_title("TF-IDF baseline confusion (row-normalised)")
plt.tight_layout(); plt.show()

---
## 5. Near-duplicate check

Variants of one scenario are meant to differ. If the generator produced near-copies,
the effective dataset is smaller than 1,800 and — worse — near-duplicates split
across train and test would inflate the test score.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(min_df=1).fit_transform(df["body"])
sim = cosine_similarity(tfidf)
np.fill_diagonal(sim, 0)

pairs = np.argwhere(sim > 0.9)
print(f"pairs with cosine > 0.90: {len(pairs) // 2}")
print(f"max off-diagonal similarity: {sim.max():.3f}")

if len(pairs):
    i, j = pairs[0]
    print(f"\nclosest pair ({sim[i, j]:.3f}):")
    print(" A:", df.iloc[i]["body"][:150])
    print(" B:", df.iloc[j]["body"][:150])

In [ ]:
# Cross-split contamination is the version of this that actually corrupts a result.
if "split" in df.columns:
    tr = df.index[df["split"] == "train"].to_numpy()
    te = df.index[df["split"] == "test"].to_numpy()
    cross = sim[np.ix_(tr, te)]
    print(f"max train-test similarity: {cross.max():.3f}")
    print(f"test emails with a >0.9 twin in train: {(cross.max(axis=0) > 0.9).sum()}")
    assert cross.max() < 0.95, "a near-duplicate spans the train/test boundary"

---
## 6. Flag ground truth

`computation_requested` and `account_specific` are recorded by the scenario spec, so
their ground truth is free. Two things to confirm before they are used as metrics:

- **Enough positives to measure.** Precision and recall on a handful of examples are
  not reportable.
- **Not confounded with class.** If every computation-demanding email were
  `tax_reliefs`, the flag's measured performance would really be the classifier's.

In [ ]:
for flag in ("computation_requested", "account_specific"):
    if flag not in df.columns:
        continue
    n = int(df[flag].sum())
    print(f"\n{flag}: {n} positives ({n/len(df):.1%})")
    spread = df[df[flag]]["label"].value_counts()
    print(spread.to_string())
    print(f"  spans {spread.size} of {df['label'].nunique()} classes")
    if spread.size < 3:
        print("  WARNING: concentrated in too few classes to separate from the classifier")

In [ ]:
# Planted PII is what scrub recall is measured against, so the plant itself must
# be verifiable. This counts what was planted; recall is computed in eval/.
if "plant_pii" in df.columns:
    planted = Counter(k for row in df["plant_pii"].dropna() for k in row)
    print("planted identifiers by type:")
    for kind, n in planted.most_common():
        print(f"  {kind:12} {n}")
    print(f"\nemails carrying planted PII: {df['plant_pii'].notna().sum()}")

---
## 7. Splits

50 / 20 / 30, stratified by class, seeded and committed. The calibration split is
separate from test on purpose: temperature scaling is *fitted* on calibration and
*reported* on test, so a calibration number is never produced from the data that
fitted it.

In [ ]:
from sklearn.model_selection import train_test_split

if "split" not in df.columns:
    idx = np.arange(len(df))
    train_idx, rest = train_test_split(idx, test_size=0.5, stratify=y, random_state=SEED)
    calib_idx, test_idx = train_test_split(
        rest, test_size=0.6, stratify=y[rest], random_state=SEED,
    )
    df.loc[train_idx, "split"] = "train"
    df.loc[calib_idx, "split"] = "calibration"
    df.loc[test_idx, "split"] = "test"

print(df["split"].value_counts().to_string())
print(f"\ncalibration examples per bucket: ~{(df['split'] == 'calibration').sum() // 5}")
print(f"test examples per class:         ~{(df['split'] == 'test').sum() // 12}")

---
## 8. Fine-tune MiniLM-L6

22M parameters, 384-dimensional embeddings. Chosen over DistilBERT on **deployment
size, not accuracy**: at ~90MB the fine-tuned artefact commits to plain git, so a
grader cloning the repository gets real weights rather than LFS pointer files
(`BUILD.md` S5.4).

The head is a linear layer over the pooled embedding, producing 12 logits. That is
what makes the confidence *native to the architecture* and therefore calibratable —
the operational argument for an encoder over an LLM classifier.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

BASE = ROOT / "models" / "all-MiniLM-L6-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = sorted(df["label"].unique())
L2I = {l: i for i, l in enumerate(LABELS)}

tokenizer = AutoTokenizer.from_pretrained(BASE, local_files_only=True)
print(f"device: {DEVICE}   labels: {len(LABELS)}")

In [ ]:
class EmailDataset(Dataset):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.texts = (frame["subject"].fillna("") + "\n\n" + frame["body"]).str.strip().tolist()
        self.labels = [L2I[l] for l in frame["label"]]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int) -> tuple[str, int]:
        return self.texts[i], self.labels[i]


def collate(batch):
    texts, labels = zip(*batch, strict=True)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=256, return_tensors="pt")
    return enc, torch.tensor(labels)


train_dl = DataLoader(EmailDataset(df[df.split == "train"]), batch_size=32,
                      shuffle=True, collate_fn=collate)
calib_dl = DataLoader(EmailDataset(df[df.split == "calibration"]), batch_size=64,
                      collate_fn=collate)
print(f"train batches {len(train_dl)}   calibration batches {len(calib_dl)}")

In [ ]:
# The architecture is imported, NOT redefined here.
#
# `triage.models.head` is the single definition, shared by this notebook, by
# scripts/train_encoder.py, and by the runtime classifier that loads the weights.
# A second copy in the notebook could drift from it, and a drifted copy would load
# the saved tensors into a differently shaped model -- producing wrong predictions
# with no error raised.
#
# Design notes that live with the definition:
#   * mean pooling over the attention mask, NOT the CLS token. MiniLM was trained
#     as a sentence encoder with mean pooling, so its CLS position never learned to
#     be a sentence representation.
#   * forward() returns raw LOGITS, not probabilities, because temperature scaling
#     divides the logits before the softmax.

from triage.models.head import MiniLMClassifier

BASE = ROOT / "models" / "all-MiniLM-L6-v2"
model = MiniLMClassifier(BASE, len(LABELS)).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"architecture from: src/triage/models/head.py  (one definition, shared)")


In [ ]:
EPOCHS = 4
optimiser = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimiser, max_lr=2e-5, total_steps=EPOCHS * len(train_dl), pct_start=0.1,
)
loss_fn = torch.nn.CrossEntropyLoss()

torch.manual_seed(SEED)
history = []

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for enc, labels in train_dl:
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        labels = labels.to(DEVICE)
        optimiser.zero_grad()
        loss = loss_fn(model(**enc), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step(); scheduler.step()
        total += loss.item()

    model.eval()
    correct = seen = 0
    with torch.no_grad():
        for enc, labels in calib_dl:
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            correct += (model(**enc).argmax(1).cpu() == labels).sum().item()
            seen += len(labels)
    row = {"epoch": epoch + 1, "train_loss": total / len(train_dl), "calib_acc": correct / seen}
    history.append(row)
    print(f"epoch {row['epoch']}  loss {row['train_loss']:.4f}  calib acc {row['calib_acc']:.4f}")

### A note on what the calibration accuracy above is, and is not

It is a **development signal** — enough to see that training converged. It is not a
result, because it comes from the split that also fits the temperature. The reported
numbers come from the test split, computed in `eval/run_eval.py`.

In [ ]:
pd.DataFrame(history).set_index("epoch").plot(subplots=True, figsize=(7, 5), marker="o")
plt.tight_layout(); plt.show()

---
## 9. Save the artefacts

Weights, the label order, and the training metadata. The label order matters as much
as the weights: a reordering would silently permute every prediction.

In [ ]:
OUT = ROOT / "models" / "classifier"
OUT.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), OUT / "model.pt")
# NOTE: state_dict stores tensors only -- no architecture. The runtime
# reads config.json from the BASE model directory and the weights from
# here, so there is one source of truth for the shape.
tokenizer.save_pretrained(OUT)
(OUT / "labels.json").write_text(json.dumps(LABELS, indent=2), encoding="utf-8")
(OUT / "training.json").write_text(json.dumps({
    "base_model": "sentence-transformers/all-MiniLM-L6-v2",
    "seed": SEED,
    "epochs": EPOCHS,
    "max_length": 256,
    "batch_size": 32,
    "learning_rate": 2e-5,
    "n_train": int((df.split == "train").sum()),
    "n_calibration": int((df.split == "calibration").sum()),
    "n_test": int((df.split == "test").sum()),
    "history": history,
}, indent=2), encoding="utf-8")

size_mb = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1024**2
print(f"saved -> {OUT.relative_to(ROOT)}  ({size_mb:.1f}MB)")
if size_mb > 95:
    print("WARNING: approaching GitHub's 100MB limit; git-lfs would be needed.")

---
## 10. What happens next, and what deliberately does not happen here

Built here: a fine-tuned encoder and a dataset that has been audited for balance,
length artefacts, vocabulary leakage and cross-split duplication.

**Not here, on purpose:**

- **Temperature scaling** — fitted on the calibration split by
  `scripts/fit_calibration.py`.
- **Every reported number** — computed by `eval/run_eval.py` on the test split and
  written to `eval/results/`. The README cites those files, never a number
  remembered from a notebook run.

The separation is the guard against the code–writeup mismatch failure mode: a figure
in the writeup can always be traced to a committed file produced by a script, rather
than to a cell someone ran once.

---
## 11. What this run produced

Recorded so a number in the writeup can be traced to the run that made it.

| | |
|---|---|
| Corpus | 1,800 emails, 150/class, 0 generation failures |
| Generator | `gpt-4o`, ~50 min, seed 42 |
| Splits | 879 train / 401 calibration / 520 test, grouped by scenario |
| Model | MiniLM-L6, 22.7M params, 87.4MB, 4 epochs, ~8 min on CPU |
| Convergence | loss 2.37 → 0.98; calibration accuracy 0.57 → 0.74 |

**Epoch 4 dipped** (0.7357 → 0.7282). On 879 training examples that is the onset of
overfitting, and it is the reason the epoch count is not raised without measuring:
a longer run would lower training loss while learning the corpus rather than the
task.

**None of the above is a result.** Calibration accuracy is a convergence signal
taken from the split that also fits the temperature. Every reported number comes
from `eval/run_eval.py` on the held-out test split and is written to
`eval/results/`, so the README can cite a committed file rather than a remembered
figure from a notebook run.